<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/tiny-transformer/blob/main/Text_Generation_with_Self_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [118]:
!pip install gensim nltk
import nltk
import gensim
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.tokenize import word_tokenize
import tensorflow as tf
import numpy as np

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [120]:
sentences = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "cats love milk",
    "dogs love bones",
    "the cat loves milk"
]

special_tokens = ["<PAD>", "<START>", "<END>"]

tokens = []
for s in sentences:
    tokens.extend(word_tokenize(s))

print(f"Tokens : {tokens}")

vocab_words = special_tokens + sorted(set(tokens))
print(f"Vocab words : {vocab_words}")

vocab = {w:i for i,w in enumerate(vocab_words)}
print(f"Vocab : {vocab}")

id2word = {i:w for w,i in vocab.items()}
print(f"id2word: {id2word}")

vocab_size = len(vocab)
print(f"Vocab size : {vocab_size}")

Tokens : ['the', 'cat', 'sat', 'on', 'the', 'mat', 'the', 'dog', 'sat', 'on', 'the', 'rug', 'cats', 'love', 'milk', 'dogs', 'love', 'bones', 'the', 'cat', 'loves', 'milk']
Vocab words : ['<PAD>', '<START>', '<END>', 'bones', 'cat', 'cats', 'dog', 'dogs', 'love', 'loves', 'mat', 'milk', 'on', 'rug', 'sat', 'the']
Vocab : {'<PAD>': 0, '<START>': 1, '<END>': 2, 'bones': 3, 'cat': 4, 'cats': 5, 'dog': 6, 'dogs': 7, 'love': 8, 'loves': 9, 'mat': 10, 'milk': 11, 'on': 12, 'rug': 13, 'sat': 14, 'the': 15}
id2word: {0: '<PAD>', 1: '<START>', 2: '<END>', 3: 'bones', 4: 'cat', 5: 'cats', 6: 'dog', 7: 'dogs', 8: 'love', 9: 'loves', 10: 'mat', 11: 'milk', 12: 'on', 13: 'rug', 14: 'sat', 15: 'the'}
Vocab size : 16


In [121]:
encoder_token = tokens
decoder_input_tokens = ["<START>"] + tokens
decoder_target_tokens = tokens + ["<END>"]

encoder_ids = [vocab[w] for w in tokens]
decoder_input_ids = [vocab[w] for w in decoder_input_tokens]
decoder_target_ids = [vocab[w] for w in decoder_target_tokens]

print(f"Size of encoder ids : {len(encoder_ids)}")
print(f"Size of decoder input ids : {len(decoder_input_ids)}")
print(f"Size of decoder target ids : {len(decoder_target_ids)}")

Size of encoder ids : 22
Size of decoder input ids : 23
Size of decoder target ids : 23


In [122]:
encoder_ids = tf.constant(encoder_ids, dtype=tf.int32)
decoder_input_ids = tf.constant(decoder_input_ids, dtype=tf.int32)
decoder_target_ids = tf.constant(decoder_target_ids, dtype=tf.int32)

In [123]:
embedding_dim = 64

embedding = tf.keras.layers.Embedding(input_dim = vocab_size, output_dim = embedding_dim)

In [124]:
def position_encoding(x):
  # Initialize the pos_encoding with zeros (token_size, embedding_dimension)
  pos_encoding = np.zeros((x.shape[0], embedding_dim))

  # fill the pos_encoding
  for i in range(x.shape[0]):
    for j in range(embedding_dim):
      if i % 2 == 0:
        pos_encoding[i,j] = np.sin(i/(10000 ** (j / embedding_dim)))
      else:
        pos_encoding[i,j] = np.cos(i/ (10000 ** ((j-1)/embedding_dim)))

  #convert to tensor
  pos_encoding = tf.cast(pos_encoding, dtype=tf.float32)
  return pos_encoding

In [125]:
encoder_emb_input = embedding(encoder_ids)
print(f"Encoder embedding shape : {encoder_emb_input.shape}")

positional_encoding = position_encoding(encoder_emb_input)
print(f"Positional Encoding shapes : {positional_encoding.shape}")

encoder_emb = encoder_emb_input + positional_encoding
print(f"Final Encoder embedding shapes : {encoder_emb.shape}")

Encoder embedding shape : (22, 64)
Positional Encoding shapes : (22, 64)
Final Encoder embedding shapes : (22, 64)


In [126]:
def self_attention(x, mask, isMaskedAttention = False):
  d = embedding_dim
  Q = tf.keras.layers.Dense(d)(x)
  K = tf.keras.layers.Dense(d)(x)
  V = tf.keras.layers.Dense(d)(x)

  scores = tf.matmul(Q, tf.transpose(K)) / tf.sqrt(tf.cast(d, dtype=tf.float32))

  if(isMaskedAttention):
    scores += mask * -1e9

  attention_weights = tf.nn.softmax(scores, axis=-1)
  return tf.matmul(attention_weights, V)

In [127]:
encoder_output = self_attention(encoder_emb, mask=[])
print(f"Encoder output shape : {encoder_output.shape}")

Encoder output shape : (22, 64)


In [128]:
decoder_emb = embedding(decoder_input_ids)
decoder_emb += position_encoding(decoder_emb)
print(f"Decoder emb shape : {decoder_emb.shape}")

seq_len = tf.shape(decoder_emb)[0]
mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)

masked_output = self_attention(decoder_emb, mask, isMaskedAttention=True)
print(f"Masked output shape : {masked_output.shape}")

Decoder emb shape : (23, 64)
Masked output shape : (23, 64)


In [129]:
Q = tf.keras.layers.Dense(embedding_dim)(masked_output)
K = tf.keras.layers.Dense(embedding_dim)(encoder_output)
V = tf.keras.layers.Dense(embedding_dim)(encoder_output)

scores = tf.matmul(Q, tf.transpose(K)) / tf.sqrt(tf.cast(embedding_dim, dtype=tf.float32))
weights = tf.nn.softmax(scores, axis=-1)
decoder_output = tf.matmul(weights, V)
print(f"Decoder output shape is {decoder_output.shape}")

Decoder output shape is (23, 64)


In [130]:
logits = tf.keras.layers.Dense(vocab_size)(decoder_output)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
loss = loss_fn(decoder_target_ids, logits)

print(f"Loss : {loss.numpy()}")

Loss : 2.8599843978881836


In [131]:
def generate_text(max_len=10):
  generated= ["<START>"]

  for _ in range(max_len):
    ids = tf.constant([vocab[w] for w in generated])
    emb = position_encoding(embedding(ids))

    Q = tf.keras.layers.Dense(embedding_dim)(emb)
    K = tf.keras.layers.Dense(embedding_dim)(encoder_output)
    V = tf.keras.layers.Dense(embedding_dim)(encoder_output)

    scores = tf.matmul(Q, tf.transpose(K)) / tf.sqrt(tf.cast(embedding_dim, dtype=tf.float32))
    weights = tf.nn.softmax(scores)
    output = tf.matmul(weights, V)

    logits= tf.keras.layers.Dense(vocab_size)(output)
    next_id = tf.argmax(logits[-1]).numpy()
    next_word = id2word[next_id]
    print(f"Predicted word : {next_word}")

    if next_word == "<END>":
      break;

    generated.append(next_word)

  return " ".join(generated[1:])

In [132]:
generate_text()

Predicted word : mat
Predicted word : the
Predicted word : dog
Predicted word : the
Predicted word : milk
Predicted word : <START>
Predicted word : loves
Predicted word : bones
Predicted word : on
Predicted word : the


'mat the dog the milk <START> loves bones on the'